# PANNs acoustic-event extraction — TAU 6000 development subset

This notebook preserves the **same PANNs extraction procedure as the original
12,291-recording notebook**, but applies it only to the fixed 6000-recording
development manifest.

Preserved settings:
- PANNs `AudioTagging` / Cnn14 via `panns_inference`
- 32 kHz mono audio
- 3-second chunks
- Top-8 AudioSet events per chunk
- recording-level aggregation by `max_score`, `mean_score`, and `count`
- `num_chunks` saved so persistence can be computed as `count / num_chunks`
- checkpoint every 25 recordings

In [1]:
from pathlib import Path
import sys
import time
import json
import numpy as np
import pandas as pd
import librosa
import torch
from IPython.display import display
from panns_inference import AudioTagging, labels
print('Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Python: C:\Users\22313906t\AppData\Local\anaconda3\envs\tau_env\python.exe
PyTorch: 2.13.0+cu126
CUDA: 12.6
CUDA available: True
GPU: NVIDIA RTX 2000 Ada Generation Laptop GPU


## 1. Paths

In [2]:
PROJECT_ROOT = Path('C:\\Users\\22313906t\\Desktop\\TAU_project')
DATA_DIR = PROJECT_ROOT / 'data_subset_6000'
MANIFEST_PATH = PROJECT_ROOT / 'manifests' / 'tau_balanced_6000_experiment_manifest.csv'
PANNS_RESULTS_DIR = PROJECT_ROOT / 'results_PANNs_6000'
PANNS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ALL_EVENTS_PATH = PANNS_RESULTS_DIR / 'tau_6000_panns_events.csv'
ALL_TOP_PATH = PANNS_RESULTS_DIR / 'tau_6000_panns_top_events.csv'
ALL_ERRORS_PATH = PANNS_RESULTS_DIR / 'tau_6000_panns_errors.csv'
CONFIG_PATH = PANNS_RESULTS_DIR / 'tau_6000_panns_config.json'
print('Project:', PROJECT_ROOT)
print('Data:', DATA_DIR)
print('Manifest:', MANIFEST_PATH)
print('PANNs results:', PANNS_RESULTS_DIR)


Project: C:\Users\22313906t\Desktop\TAU_project
Data: C:\Users\22313906t\Desktop\TAU_project\data_subset_6000
Manifest: C:\Users\22313906t\Desktop\TAU_project\manifests\tau_balanced_6000_experiment_manifest.csv
PANNs results: C:\Users\22313906t\Desktop\TAU_project\results_PANNs_6000


## 2. Fixed PANNs settings

In [3]:
PANNS_SR = 32000
CHUNK_SECONDS = 3.0
TOP_K = 8
CHECKPOINT_EVERY = 25
PANNS_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
config = {
    'audio_model': 'PANNs Cnn14',
    'package': 'panns-inference',
    'sample_rate': PANNS_SR,
    'chunk_seconds': CHUNK_SECONDS,
    'top_k_per_chunk': TOP_K,
    'device': PANNS_DEVICE,
    'subset': 'TAU balanced development 6000',
}
with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)
print('PANNs device:', PANNS_DEVICE)
print('PANNs sample rate:', PANNS_SR)
print('Chunk seconds:', CHUNK_SECONDS)
print('Top K per chunk:', TOP_K)


PANNs device: cuda
PANNs sample rate: 32000
Chunk seconds: 3.0
Top K per chunk: 8


## 3. Load the exact 6000-recording manifest

The original notebook scanned the complete 12,291-WAV collection.  
For the reproducibility rerun, the manifest is authoritative: only these 6000
development recordings are processed.

In [4]:
manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df['clip_id'] = manifest_df['clip_id'].astype(str)
manifest_df['ground_truth'] = manifest_df['ground_truth'].astype(str)
print('Manifest rows:', len(manifest_df))
print('Unique clip IDs:', manifest_df['clip_id'].nunique())
display(manifest_df['ground_truth'].value_counts().to_frame('number_of_clips'))
display(manifest_df.head())


Manifest rows: 6000
Unique clip IDs: 6000


,number_of_clips
ground_truth,
metro,600
shopping mall,600
airport,600
metro station,600
street traffic,600
bus,600
park,600
public square,600
tram,600


,clip_id,ground_truth,audio_path,video_path
0,metro-lisbon-1055-41182,metro,C:\Users\22313906t\Desktop\TAU_project\data_su...,C:\Users\22313906t\Desktop\TAU_project\data_su...
1,shopping_mall-barcelona-125-3702,shopping mall,C:\Users\22313906t\Desktop\TAU_project\data_su...,C:\Users\22313906t\Desktop\TAU_project\data_su...
2,airport-lisbon-1122-41270,airport,C:\Users\22313906t\Desktop\TAU_project\data_su...,C:\Users\22313906t\Desktop\TAU_project\data_su...
3,metro_station-stockholm-239-7087,metro station,C:\Users\22313906t\Desktop\TAU_project\data_su...,C:\Users\22313906t\Desktop\TAU_project\data_su...
4,street_traffic-prague-1193-44472,street traffic,C:\Users\22313906t\Desktop\TAU_project\data_su...,C:\Users\22313906t\Desktop\TAU_project\data_su...


## 4. Resolve audio paths

In [5]:
wav_index = None


def build_wav_index():
    wav_files = sorted((p for p in DATA_DIR.rglob('*.wav') if p.is_file()))
    index = {}
    for p in wav_files:
        index.setdefault(p.stem, []).append(p)
    print('WAV files indexed:', len(wav_files))
    return index


def resolve_audio_path(row):
    global wav_index
    if 'audio_path' in manifest_df.columns and pd.notna(row.get('audio_path')):
        candidate = Path(str(row['audio_path']))
        if candidate.exists():
            return candidate
    if wav_index is None:
        wav_index = build_wav_index()
    key = Path(str(row['clip_id'])).stem
    matches = wav_index.get(key, [])
    if len(matches) == 1:
        return matches[0]
    if len(matches) == 0:
        raise FileNotFoundError(f"No WAV found for clip_id={row['clip_id']}")
    raise RuntimeError(f"Multiple WAV files found for clip_id={row['clip_id']}: {matches}")


audio_rows = []
for _, row in manifest_df.iterrows():
    p = resolve_audio_path(row)
    audio_rows.append({'clip_id': str(row['clip_id']), 'ground_truth': str(row['ground_truth']), 'audio_path': str(p)})
audio_df = pd.DataFrame(audio_rows)
print('Selected audio recordings:', len(audio_df))
display(audio_df.head())


Selected audio recordings: 6000


,clip_id,ground_truth,audio_path
0,metro-lisbon-1055-41182,metro,C:\Users\22313906t\Desktop\TAU_project\data_su...
1,shopping_mall-barcelona-125-3702,shopping mall,C:\Users\22313906t\Desktop\TAU_project\data_su...
2,airport-lisbon-1122-41270,airport,C:\Users\22313906t\Desktop\TAU_project\data_su...
3,metro_station-stockholm-239-7087,metro station,C:\Users\22313906t\Desktop\TAU_project\data_su...
4,street_traffic-prague-1193-44472,street traffic,C:\Users\22313906t\Desktop\TAU_project\data_su...


## 5. Load PANNs

In [6]:
print('Loading PANNs...')
panns_model = AudioTagging(checkpoint_path=None, device=PANNS_DEVICE)
print('PANNs loaded.')
print('Number of PANNs labels:', len(labels))


Loading PANNs...
Checkpoint path: C:\Users\22313906t/panns_data/Cnn14_mAP=0.431.pth


C:\Users\22313906t\AppData\Local\anaconda3\envs\tau_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU number: 1
PANNs loaded.
Number of PANNs labels: 527


## 6. Same 3-second chunking as the original notebook

In [7]:
def chunk_audio(y, sr=32000, chunk_seconds=3.0):
    chunk_size = int(sr * chunk_seconds)
    if chunk_size <= 0:
        raise ValueError('chunk_seconds must be > 0')
    chunks = []
    for start_idx in range(0, len(y), chunk_size):
        end_idx = min(start_idx + chunk_size, len(y))
        chunk = y[start_idx:end_idx]
        if len(chunk) == 0:
            continue
        chunks.append({'chunk_index': len(chunks), 'start_s': start_idx / sr, 'end_s': end_idx / sr, 'audio': chunk})
    return chunks


## 7. Same Top-8 PANNs inference per chunk

In [8]:
def classify_panns_chunk(chunk, top_k=8):
    chunk = np.asarray(chunk, dtype=np.float32)
    audio_batch = chunk[np.newaxis, :]
    clipwise_output, embedding = panns_model.inference(audio_batch)
    scores = clipwise_output[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    predictions = []
    for index in top_indices:
        predictions.append({'label': str(labels[index]), 'score': float(scores[index])})
    return predictions


## 8. Same recording-level aggregation as the original notebook

In [9]:
def classify_panns_audio_file(audio_path, top_k=8, chunk_seconds=3.0, sr=32000):
    audio_path = str(audio_path)
    y, _ = librosa.load(audio_path, sr=sr, mono=True)
    y = np.asarray(y, dtype=np.float32)
    chunks = chunk_audio(y=y, sr=sr, chunk_seconds=chunk_seconds)
    chunk_predictions = []
    for chunk_info in chunks:
        predictions = classify_panns_chunk(chunk=chunk_info['audio'], top_k=top_k)
        chunk_predictions.append({'chunk_index': chunk_info['chunk_index'], 'start_s': chunk_info['start_s'], 'end_s': chunk_info['end_s'], 'labels': predictions})
    label_scores = {}
    for chunk in chunk_predictions:
        for item in chunk['labels']:
            label = item['label']
            score = float(item['score'])
            label_scores.setdefault(label, [])
            label_scores[label].append(score)
    summary = []
    for label, scores in label_scores.items():
        summary.append({'label': label, 'max_score': float(max(scores)), 'mean_score': float(np.mean(scores)), 'count': int(len(scores))})
    summary.sort(key=lambda x: (x['max_score'], x['mean_score'], x['count']), reverse=True)
    return {
        'audio_path': audio_path,
        'num_chunks': len(chunks),
        'chunks': chunk_predictions,
        'summary': summary,
    }


## 9. One-recording test

In [10]:
test_row = audio_df.iloc[0]
test_result = classify_panns_audio_file(
    audio_path=test_row['audio_path'],
    top_k=TOP_K,
    chunk_seconds=CHUNK_SECONDS,
    sr=PANNS_SR,
)
print('Clip:', test_row['clip_id'])
print('Chunks:', test_result['num_chunks'])
display(pd.DataFrame(test_result['summary'][:10]))


Clip: metro-lisbon-1055-41182
Chunks: 4


,label,max_score,mean_score,count
0,Speech,0.706648,0.510619,4
1,Train,0.597942,0.425186,4
2,Vehicle,0.523056,0.479815,4
3,Rail transport,0.436387,0.266602,4
4,"Railroad car, train wagon",0.403455,0.230627,4
5,"Subway, metro, underground",0.267064,0.176796,3
6,Field recording,0.149755,0.112956,2
7,Aircraft,0.093867,0.082256,2
8,Music,0.084393,0.084393,1
9,"Fixed-wing aircraft, airplane",0.078666,0.078666,1


## 10. Fresh run or resume

Set `FRESH_RUN = True` only when you intentionally want to delete old 6000
PANNs outputs and rerun from zero. Otherwise leave it `False`.

In [11]:
FRESH_RUN = False
if FRESH_RUN:
    for path in [ALL_EVENTS_PATH, ALL_TOP_PATH, ALL_ERRORS_PATH]:
        if path.exists():
            path.unlink()
            print('Deleted:', path)
    print('Fresh PANNs run ready.')
else:
    print('Resume mode: existing outputs will be kept.')


Resume mode: existing outputs will be kept.


## 11. Resume/checkpoint setup

In [12]:
if ALL_EVENTS_PATH.exists():
    existing_events_df = pd.read_csv(ALL_EVENTS_PATH)
else:
    existing_events_df = pd.DataFrame()
if ALL_TOP_PATH.exists():
    existing_top_df = pd.read_csv(ALL_TOP_PATH)
    processed_ids = set(existing_top_df['clip_id'].astype(str))
else:
    existing_top_df = pd.DataFrame()
    processed_ids = set()
if ALL_ERRORS_PATH.exists():
    existing_errors_df = pd.read_csv(ALL_ERRORS_PATH)
else:
    existing_errors_df = pd.DataFrame()
remaining_df = audio_df[~audio_df['clip_id'].astype(str).isin(processed_ids)].copy().reset_index(drop=True)
print('Total clips:', len(audio_df))
print('Already processed:', len(processed_ids))
print('Remaining:', len(remaining_df))


Total clips: 6000
Already processed: 0
Remaining: 6000


In [13]:
new_event_rows = []
new_top_rows = []
new_error_rows = []


def save_panns_checkpoint():
    global existing_events_df
    global existing_top_df
    global existing_errors_df
    global new_event_rows
    global new_top_rows
    global new_error_rows
    if new_event_rows:
        temp = pd.DataFrame(new_event_rows)
        existing_events_df = pd.concat([existing_events_df, temp], ignore_index=True)
        existing_events_df = existing_events_df.drop_duplicates(subset=['clip_id', 'label'], keep='last')
        existing_events_df.to_csv(ALL_EVENTS_PATH, index=False)
        new_event_rows = []
    if new_top_rows:
        temp = pd.DataFrame(new_top_rows)
        existing_top_df = pd.concat([existing_top_df, temp], ignore_index=True)
        existing_top_df = existing_top_df.drop_duplicates(subset=['clip_id'], keep='last')
        existing_top_df.to_csv(ALL_TOP_PATH, index=False)
        new_top_rows = []
    if new_error_rows:
        temp = pd.DataFrame(new_error_rows)
        existing_errors_df = pd.concat([existing_errors_df, temp], ignore_index=True)
        existing_errors_df = existing_errors_df.drop_duplicates(subset=['clip_id'], keep='last')
        existing_errors_df.to_csv(ALL_ERRORS_PATH, index=False)
        new_error_rows = []


## 12. Run PANNs on the 6000 development recordings

In [14]:
processed_since_checkpoint = 0
start_time = time.time()
total_remaining = len(remaining_df)
for idx, row in remaining_df.iterrows():
    clip_id = str(row['clip_id'])
    ground_truth = str(row['ground_truth'])
    current_audio_path = str(row['audio_path'])
    print(f'[{idx + 1}/{total_remaining}] {ground_truth:20s} | {clip_id}')
    try:
        result = classify_panns_audio_file(
            audio_path=current_audio_path,
            top_k=TOP_K,
            chunk_seconds=CHUNK_SECONDS,
            sr=PANNS_SR,
        )
        for event in result['summary']:
            new_event_rows.append({'clip_id': clip_id, 'ground_truth': ground_truth, 'label': event['label'], 'max_score': event['max_score'], 'mean_score': event['mean_score'], 'count': event['count'], 'num_chunks': result['num_chunks']})
        top_row = {'clip_id': clip_id, 'ground_truth': ground_truth, 'num_chunks': result['num_chunks']}
        for rank, event in enumerate(result['summary'][:10], start=1):
            top_row[f'event_{rank}'] = event['label']
            top_row[f'score_{rank}'] = event['max_score']
        new_top_rows.append(top_row)
    except Exception as e:
        print('ERROR:', clip_id, str(e))
        new_error_rows.append({'clip_id': clip_id, 'ground_truth': ground_truth, 'audio_path': current_audio_path, 'error': str(e)})
    processed_since_checkpoint += 1
    if processed_since_checkpoint >= CHECKPOINT_EVERY:
        save_panns_checkpoint()
        processed_since_checkpoint = 0
        print('----- PANNs CHECKPOINT SAVED -----')
save_panns_checkpoint()
elapsed_minutes = (time.time() - start_time) / 60
print('\nPANNs processing complete.')
print('Elapsed minutes:', round(elapsed_minutes, 2))
print('\nEvents:', ALL_EVENTS_PATH)
print('Top events:', ALL_TOP_PATH)
if ALL_ERRORS_PATH.exists():
    print('Errors:', ALL_ERRORS_PATH)


[1/6000] metro                | metro-lisbon-1055-41182
[2/6000] shopping mall        | shopping_mall-barcelona-125-3702
[3/6000] airport              | airport-lisbon-1122-41270
[4/6000] metro station        | metro_station-stockholm-239-7087
[5/6000] street traffic       | street_traffic-prague-1193-44472
[6/6000] bus                  | bus-lyon-1124-41154
[7/6000] bus                  | bus-helsinki-19-776
[8/6000] airport              | airport-lisbon-1175-44703
[9/6000] metro                | metro-lyon-1206-44910
[10/6000] airport              | airport-london-205-6192
[11/6000] street traffic       | street_traffic-london-270-8216
[12/6000] park                 | park-lyon-1188-44569
[13/6000] bus                  | bus-lisbon-1212-45534
[14/6000] airport              | airport-paris-7-345
[15/6000] public square        | public_square-lyon-1017-41596
[16/6000] park                 | park-lisbon-1057-42931
[17/6000] metro station        | metro_station-barcelona-229-6893
[18/600

## 13. Add persistence and inspect final output

In [15]:
all_panns_events_df = pd.read_csv(ALL_EVENTS_PATH)
all_panns_top_df = pd.read_csv(ALL_TOP_PATH)
all_panns_events_df[
    'persistence'] = pd.to_numeric(all_panns_events_df['count'],
    errors='coerce') / pd.to_numeric(all_panns_events_df['num_chunks'],
    errors='coerce').replace(0,
    np.nan,
)

# Save persistence in the canonical 6000 event file as a convenience.
all_panns_events_df.to_csv(ALL_EVENTS_PATH, index=False)
print('Processed clips:', all_panns_top_df['clip_id'].astype(str).nunique())
print('PANNs event rows:', len(all_panns_events_df))
display(all_panns_events_df.head(20))


Processed clips: 6000
PANNs event rows: 86665


,clip_id,ground_truth,label,max_score,mean_score,count,num_chunks,persistence
0,metro-lisbon-1055-41182,metro,Speech,0.706648,0.510619,4,4,1.00
1,metro-lisbon-1055-41182,metro,Train,0.597942,0.425186,4,4,1.00
2,metro-lisbon-1055-41182,metro,Vehicle,0.523056,0.479815,4,4,1.00
3,metro-lisbon-1055-41182,metro,Rail transport,0.436387,0.266602,4,4,1.00
4,metro-lisbon-1055-41182,metro,"Railroad car, train wagon",0.403455,0.230627,4,4,1.00
5,metro-lisbon-1055-41182,metro,"Subway, metro, underground",0.267064,0.176796,3,4,0.75
6,metro-lisbon-1055-41182,metro,Field recording,0.149755,0.112956,2,4,0.50
7,metro-lisbon-1055-41182,metro,Aircraft,0.093867,0.082256,2,4,0.50
8,metro-lisbon-1055-41182,metro,Music,0.084393,0.084393,1,4,0.25
9,metro-lisbon-1055-41182,metro,"Fixed-wing aircraft, airplane",0.078666,0.078666,1,4,0.25
